# Beginner RAG on Kaggle

Same idea as the local app: **PDF (text + tables) → chunks → embeddings → ChromaDB → ask**.

### Before you run
1. **Settings → Internet → On** (needed for `pip` and the embedding-model download).
2. Run cells **top to bottom**.
3. Optional: upload your own PDF via *Add data* / the file browser, then set `PDF_PATH` in the config cell.


## 1. Install packages

Kaggle images change over time — reinstalling keeps this notebook self-contained.


In [ ]:
# Internet must be ON in the notebook Settings panel.
!pip install -q pdfplumber chromadb sentence-transformers reportlab

print("Install done.")


## 2. Config

Paths under `/kaggle/working` survive for the session and are writable.


In [ ]:
from pathlib import Path

WORK = Path("/kaggle/working/rag_demo")
WORK.mkdir(parents=True, exist_ok=True)

CHROMA_DIR = WORK / "chroma"
SAMPLE_PDF = WORK / "sample_company_report.pdf"

# Point this at your uploaded PDF if you have one, e.g.:
# PDF_PATH = Path("/kaggle/input/your-dataset/your.pdf")
PDF_PATH = SAMPLE_PDF

QUESTION = "What was total 2023 revenue and which product grew fastest?"
TOP_K = 4
CHUNK_SIZE = 500
CHUNK_OVERLAP = 100
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

print("Working dir:", WORK)
print("PDF_PATH:", PDF_PATH)


## 3. Create a tiny sample PDF (text + table)

Skipped automatically if `PDF_PATH` already exists.


In [ ]:
from reportlab.lib import colors
from reportlab.lib.pagesizes import letter
from reportlab.lib.units import inch
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle
from reportlab.lib.styles import getSampleStyleSheet

if PDF_PATH.exists():
    print("Using existing PDF:", PDF_PATH)
else:
    doc = SimpleDocTemplate(str(SAMPLE_PDF), pagesize=letter)
    styles = getSampleStyleSheet()
    story = [
        Paragraph("Acme Widgets Annual Report 2023", styles["Title"]),
        Spacer(1, 0.2 * inch),
        Paragraph(
            "Acme Widgets designs and sells modular dashboard widgets for small businesses. "
            "In 2023 the company expanded into two new regions: the Pacific Northwest and "
            "Northern Europe. Customer support response time improved from 18 hours to 6 hours.",
            styles["BodyText"],
        ),
        Spacer(1, 0.25 * inch),
        Paragraph("Revenue by Product Line", styles["Heading2"]),
        Paragraph(
            "The table below summarizes revenue (in millions USD) for each product line.",
            styles["BodyText"],
        ),
        Spacer(1, 0.15 * inch),
    ]
    data = [
        ["Product", "2022 Revenue", "2023 Revenue", "Growth"],
        ["Core Widgets", "12.4", "15.1", "21.8%"],
        ["Analytics Pack", "4.2", "6.8", "61.9%"],
        ["Support Plus", "2.1", "2.9", "38.1%"],
        ["Total", "18.7", "24.8", "32.6%"],
    ]
    table = Table(data, colWidths=[1.6 * inch, 1.4 * inch, 1.4 * inch, 1.1 * inch])
    table.setStyle(
        TableStyle(
            [
                ("BACKGROUND", (0, 0), (-1, 0), colors.Color(0.12, 0.42, 0.29)),
                ("TEXTCOLOR", (0, 0), (-1, 0), colors.whitesmoke),
                ("GRID", (0, 0), (-1, -1), 0.5, colors.grey),
                ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
                ("BACKGROUND", (0, -1), (-1, -1), colors.Color(0.9, 0.95, 0.92)),
                ("ALIGN", (1, 0), (-1, -1), "CENTER"),
            ]
        )
    )
    story.append(table)
    story.append(Spacer(1, 0.3 * inch))
    story.append(Paragraph("Outlook", styles["Heading2"]))
    story.append(
        Paragraph(
            "For 2024, Acme Widgets plans to launch a mobile companion app and hire 40 "
            "additional engineers. The CEO stated that Analytics Pack is expected to "
            "overtake Core Widgets as the fastest-growing line.",
            styles["BodyText"],
        )
    )
    doc.build(story)
    PDF_PATH = SAMPLE_PDF
    print("Wrote sample PDF:", PDF_PATH)


## 4. Parse PDF — text **and** tables

Tables become simple markdown-like text so they can be embedded like any other passage.


In [ ]:
import pdfplumber

def table_to_text(table):
    if not table:
        return ""
    rows = []
    for row in table:
        cells = [(c or "").strip().replace("\n", " ") for c in row]
        if any(cells):
            rows.append(cells)
    if not rows:
        return ""
    lines = [" | ".join(rows[0]), "---"]
    for row in rows[1:]:
        lines.append(" | ".join(row))
    return "\n".join(lines)

documents = []  # each: {content, metadata}
with pdfplumber.open(PDF_PATH) as pdf:
    for page_num, page in enumerate(pdf.pages, start=1):
        text = (page.extract_text() or "").strip()
        if text:
            documents.append(
                {
                    "content": text,
                    "metadata": {"source": PDF_PATH.name, "page": page_num, "type": "text"},
                }
            )
        for i, table in enumerate(page.extract_tables() or [], start=1):
            rendered = table_to_text(table)
            if rendered:
                documents.append(
                    {
                        "content": f"[Table {i} on page {page_num}]\n{rendered}",
                        "metadata": {"source": PDF_PATH.name, "page": page_num, "type": "table"},
                    }
                )

print(f"Documents extracted: {len(documents)}")
for d in documents:
    print(f"  - page {d['metadata']['page']} ({d['metadata']['type']}): {len(d['content'])} chars")


## 5. Chunk the documents

Small overlapping chunks make retrieval more precise.


In [ ]:
def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    text = (text or "").strip()
    if not text:
        return []
    chunks_out, start = [], 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        piece = text[start:end].strip()
        if piece:
            chunks_out.append(piece)
        if end >= len(text):
            break
        start = end - overlap
    return chunks_out

chunks = []
for doc in documents:
    for i, piece in enumerate(chunk_text(doc["content"])):
        chunks.append(
            {
                "content": piece,
                "metadata": {**doc["metadata"], "chunk_index": i},
            }
        )

print(f"Chunks: {len(chunks)}")


## 6. Embed + store in ChromaDB

Local free embeddings (`all-MiniLM-L6-v2`). First run downloads the model (~80MB).


In [ ]:
import uuid
import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer

# Fresh collection each run so re-runs are predictable
if CHROMA_DIR.exists():
    import shutil
    shutil.rmtree(CHROMA_DIR)
CHROMA_DIR.mkdir(parents=True, exist_ok=True)

model = SentenceTransformer(EMBED_MODEL, device="cpu")
print("Embedding model ready.")

embeddings = model.encode(
    [c["content"] for c in chunks],
    show_progress_bar=True,
    normalize_embeddings=True,
).tolist()

client = chromadb.PersistentClient(
    path=str(CHROMA_DIR),
    settings=Settings(anonymized_telemetry=False),
)
collection = client.get_or_create_collection(
    name="pdf_chunks",
    metadata={"hnsw:space": "cosine"},
)

collection.add(
    ids=[str(uuid.uuid4()) for _ in chunks],
    documents=[c["content"] for c in chunks],
    metadatas=[
        {
            "source": str(c["metadata"]["source"]),
            "page": int(c["metadata"]["page"]),
            "type": str(c["metadata"]["type"]),
            "chunk_index": int(c["metadata"]["chunk_index"]),
        }
        for c in chunks
    ],
    embeddings=embeddings,
)

print("Indexed chunks:", collection.count())


## 7. Ask a question (retrieve top sources)

This is retrieval-only — no paid LLM key required. You still see the passages RAG would feed to a generator.


In [ ]:
query_vec = model.encode(
    [QUESTION], show_progress_bar=False, normalize_embeddings=True
).tolist()[0]

results = collection.query(
    query_embeddings=[query_vec],
    n_results=min(TOP_K, collection.count()),
    include=["documents", "metadatas", "distances"],
)

print("Question:", QUESTION)
print()
docs = results["documents"][0]
metas = results["metadatas"][0]
dists = results["distances"][0]

print("Answer (retrieval-only): most relevant passages\n")
for i, (content, meta, dist) in enumerate(zip(docs, metas, dists), start=1):
    print(f"{i}. {meta.get('source')} | page {meta.get('page')} | {meta.get('type')} | distance={dist:.3f}")
    print(content)
    print()


## Tips

- **Internet off?** Turn it on, then re-run the install + model cells.
- **Your own PDF:** upload it, set `PDF_PATH`, re-run from the parse cell.
- **Scanned PDFs:** this notebook has no OCR — use a text-based PDF.
- **Same stack locally:** see the repo `README.md` for the FastAPI + React app.
